In [17]:
import os
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.padding import PKCS7
from cryptography.hazmat.backends import default_backend

## 1. Symmetric Layer (AES-CBC) for large files.

Vi implementerer funktioner til symmetrisk kryptering og afkryptering af data.

AES = Advanced Encryption Standard

Cipher Block Chaining (CBC)

AES opererer i blokke af bestemte størrelser. Derfor benyttes PKCS7 til padding.

In [18]:
def encrypt_data(data, aes_key):
    iv = os.urandom(16)

    padder = PKCS7(128).padder()
    padded_data = padder.update(data) + padder.finalize()

    cipher = Cipher(
        algorithms.AES(aes_key),
        modes.CBC(iv),
        backend=default_backend()
    )
    encryptor = cipher.encryptor()
    ciphertext = encryptor.update(padded_data) + encryptor.finalize()

    return iv, ciphertext

def decrypt_data(iv, ciphertext, aes_key):
    cipher = Cipher(
        algorithms.AES(aes_key),
        modes.CBC(iv),
        backend=default_backend()
    )
    decryptor = cipher.decryptor()
    padded_data = decryptor.update(ciphertext) + decryptor.finalize()

    unpadder = PKCS7(128).unpadder()
    data = unpadder.update(padded_data) + unpadder.finalize()

    return data

Vi kan prøve at kryptere og afkryptere en data pakke.

In [3]:
aes_key = os.urandom(32)
raw_data = b"HowAreYouDoing?"
iv, ciphertext = encrypt_data(raw_data, aes_key)
print(f"Before encryption: {raw_data}")
print(f"After encryption: iv={iv} ciphertext={ciphertext}")

decrypt_data = decrypt_data(iv, ciphertext, aes_key)
print(f"After decryption: {decrypt_data}")

Before encryption: b'HowAreYouDoing?'
After encryption: iv=b'%\n\xbb\xd7\x7f\xdf\x98Y3u\x1br\xed\x01Q*' ciphertext=b'\xe6,f\xae\x9a\xdd\xb3\xe6.\xbbqv-\x0e+"'
After decryption: b'HowAreYouDoing?'


## 2. Assymetric layer (RSA-OAEP)

Vi implementerer funktioner til asymmetrisk kryptering og afkryptering af data.

RSA = Rivest–Shamir–Adleman

OAEP = Optimal Asymmetric Encryption Padding

Her benyttes en Public og Private nøgle til at beskytte AES nøglen, så kun brugeren med private nøgle kan benytte den.

### 2.1 Generate RSA Keys

In [19]:
def generate_rsa_keys():
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )
    public_key = private_key.public_key()
    return private_key, public_key

De generede nøgler er objekter.

In [6]:
pub_key, pri_key = generate_rsa_keys()
print(pub_key)
print(pri_key)

### 2.2 Wrap using RSA-OAEP

Vi implementerer metoder til at wrappe og unwrappe AES nøglen.

In [20]:
def wrap_key(aes_key, public_key):
    encrypted_key = public_key.encrypt(
        aes_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    return encrypted_key


def unwrap_key(encrypted_key, private_key):
    aes_key = private_key.decrypt(
        encrypted_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    return aes_key

Vi prøver at wrappe og unwrappe en AES nøgle med metoderne.

In [8]:
aes_key = os.urandom(32)
private, public = generate_rsa_keys()

encrypted_key = wrap_key(aes_key,public)
decrypted_key = unwrap_key(encrypted_key,private)

print(f"Initial AES key\n{aes_key}\n\nEncrypted key\n{encrypted_key}\n\nDecrypted key\n{decrypted_key}")

Initial AES key
b'+\xe8\x89ANCA,t\xf4U\x05$v\xf9\xd3\x8aw\xfd\x90\xfa\xeat\xa9\x01\xbe,\xc0\xbbO\x04\xeb'

Encrypted key
b'kt_\x90\x12\xfbY\xe7\x9fc\xa0\xb6\xce\xc5\\\x9d)\x8a\xbe\xc5t6\xc2\xf9%\xf8\xbd\xcb\xa9\x95\xd1\xee\x12\x04\xc2EE\x81\x06\xfb\xa8\xe2\xb9\x05\x86\xd3*{Uu\x0f\xee}\xe8\xf8\xd6i\xdc\xc8<\xe1G\xbeXcu\x1bY\xbb\xd3v)\x01\x9f0\x9b\xc9\xa1\xc8Z\xd0#\xd4\x1a\xb9B\\M\x16[\xe7\n\xb1\x8ah6\xa8\x00\x1a\xba\x01\xf2\x10\xb5u\xff\xec\x9aD<q?\xd1i\x1d\x02#\x83\x0c\x88-\xfc\xf7\xe6\x8d\xce\x1bNi\xb7%\xea\xcb\xa5\xd0\xbc\x9f\rXE\x14\x8a\xe0Ik\xf4\xb6\xef\xde\xbfE\x82?\xba\x81&\x81\xbfHM\x1cJ\xa2\xfb\xe8\xbb\x8c\xc0\x07\xb9\xda\xf5\xbc\xa8\x1fq\xc8\xe73\x02f\xad\xaa\xe1+\xb3t\x0f\x8d\xc5\x01\x95\xa4\x1d\x97_\xe2U\xb5\xdd:c\x88\x89"S+\xbc<\xed\xf6\xe5,\xf6\x11\xf6\xc8\xd7\xc9M7{FLR\x156y\x0e_\xa9\xefq\x93\n\xf5N\xc6"BJb\x97\x1f\x1a\xb2\xf5\x03j\xda,Y\xb1s.\xf9'

Decrypted key
b'+\xe8\x89ANCA,t\xf4U\x05$v\xf9\xd3\x8aw\xfd\x90\xfa\xeat\xa9\x01\xbe,\xc0\xbbO\x04\xeb'


## 4. Integrity layer (RSA-PSS).

PSS (Probabilistic Signature Scheme) bruges til at sikre Integrity (data er uændret) og Authenticity (data kommer fra troværdig bruger).
Man siger at den krypterede pakke "signeres digitale".

Hvis data ændres eller bruger ikke er troværdig (ikke har private nøgle) returneres "False".

In [21]:
def sign_package(private_key, data):
    signature = private_key.sign(
        data,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    return signature


def verify_signature(public_key, data, signature):
    try:
        public_key.verify(
            signature,
            data,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )
        return True
    except:
        return False

Vi tester hvad der sker, hvis data ikke er uændret.

In [10]:
aes_key = os.urandom(32)
raw_data = b"minMegetStaerkeKode"
corrputed_data = b"minMeget_StaerkeKode"

signature = sign_package(private, raw_data)

verification_no_errors = verify_signature(public, raw_data, signature)
print(verification_no_errors)
verification_corrupted_data = verify_signature(public, raw_data, corrputed_data)
print(verification_corrupted_data)

True
False


## 5. Data Vault Process

Vi samler alle delene af vores Vault i to funktioner.

create_vault() - Krypterer data, wrapper AES nøgle og signerer med PSS.

open_vault() - Verificerer data og bruger, unwrapper AES nøgle og afkrypterer data.

In [22]:
def create_vault(data, public_key, signing_private_key):
    aes_key = os.urandom(32)

    iv, ciphertext = encrypt_data(data, aes_key)

    wrapped_key = wrap_key(aes_key, public_key)

    package = wrapped_key + iv + ciphertext

    signature = sign_package(signing_private_key, package)

    return wrapped_key, iv, ciphertext, signature


def open_vault(wrapped_key, iv, ciphertext, signature, private_key, verify_public_key):
    package = wrapped_key + iv + ciphertext

    if not verify_signature(verify_public_key, package, signature):
        raise Exception("❌ Integrity check failed! Data corrupted or tampered.")

    print("✅ Signature verified.")

    aes_key = unwrap_key(wrapped_key, private_key)

    data = decrypt_data(iv, ciphertext, aes_key)

    return data

## Example

Vi tester at vores Vault kan kryptere og afkryptere data.

In [23]:
# Simulated biomedical data (ECG)
biomedical_data = "minMegetStaerkeKode".encode("utf-8")

# Generate keys
receiver_private, receiver_public = generate_rsa_keys()
signer_private, signer_public = generate_rsa_keys()

# Create vault
wrapped_key, iv, ciphertext, signature = create_vault(
biomedical_data,
receiver_public,
signer_private
)

print("🔒 Data stored securely.")

# Open vault
decrypted_data = open_vault(
    wrapped_key,
    iv,
    ciphertext,
    signature,
    receiver_private,
    signer_public
)

print("🔓 Decrypted Data:", decrypted_data)

🔒 Data stored securely.
✅ Signature verified.
🔓 Decrypted Data: b'minMegetStaerkeKode'
